In [10]:
import pandas as pd
import os
from pathlib import Path
import csv
from pathlib import Path

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [11]:
dataset_specs = {
    "obis_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OBIS\obis_final.parquet"),
    },
    "gbif_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet"),
    },
    "ogsl_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OGSL\ogsl_final.parquet"),
    },
    "cb_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\CB\cb_final.parquet"),
    },
}

def normalize_type(value):
    t = str(value).strip().lower()
    if any(x in t for x in ["int", "integer", "smallint", "bigint"]):
        return "integer"
    if any(x in t for x in ["float", "double", "decimal", "numeric", "real"]):
        return "float"
    if any(x in t for x in ["datetime", "timestamp", "date", "time"]):
        return "datetime"
    if "bool" in t:
        return "boolean"
    if any(x in t for x in ["category", "categorical"]):
        return "category"
    if any(x in t for x in ["string", "object", "char", "text", "varchar"]):
        return "string"
    return t

# Load all parquets
parquets = {}
schema_rows = []
comparison_rows = []

for dataset, spec in dataset_specs.items():
    parquet_path = spec["parquet"]
    if not parquet_path.exists():
        schema_rows.append({"dataset": dataset, "status": "missing parquet"})
        continue
    
    df = pd.read_parquet(parquet_path)
    parquets[dataset] = df
    schema_rows.append({
        "dataset": dataset,
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "status": "loaded"
    })

# Compare parquets to each other
datasets = list(parquets.keys())
all_columns = set()
for df in parquets.values():
    all_columns.update(df.columns)

for col in sorted(all_columns):
    for dataset1, dataset2 in [(datasets[i], datasets[j]) for i in range(len(datasets)) for j in range(i+1, len(datasets))]:
        df1 = parquets[dataset1]
        df2 = parquets[dataset2]
        
        in_df1 = col in df1.columns
        in_df2 = col in df2.columns
        
        if in_df1 and in_df2:
            type1 = normalize_type(str(df1[col].dtype))
            type2 = normalize_type(str(df2[col].dtype))
            if type1 != type2:
                comparison_rows.append({
                    "column": col,
                    "dataset1": dataset1,
                    "type1": str(df1[col].dtype),
                    "dataset2": dataset2,
                    "type2": str(df2[col].dtype),
                    "issue": "dtype mismatch"
                })
        elif in_df1 and not in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset1} only"
            })
        elif not in_df1 and in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset2} only"
            })

schema_report = pd.DataFrame(schema_rows)
comparison_report = pd.DataFrame(comparison_rows)

# Save to CSV
comparison_report.to_csv("parquet_comparison_report.csv", index=False)
schema_report.to_csv("parquet_schema_report.csv", index=False)

print(f"Saved comparison_report to: parquet_comparison_report.csv")
print(f"Saved schema_report to: parquet_schema_report.csv")

schema_report, comparison_report

Saved comparison_report to: parquet_comparison_report.csv
Saved schema_report to: parquet_schema_report.csv


(      dataset   n_rows  n_columns  status
 0  obis_final  1533558         67  loaded
 1  gbif_final  2588475         67  loaded
 2  ogsl_final     7257         68  loaded
 3    cb_final   243722         68  loaded,
         column    dataset1    type1    dataset2    type2           issue
 0      aphiaID  obis_final    int32  gbif_final   object  dtype mismatch
 1      aphiaID  obis_final    int32    cb_final  float64  dtype mismatch
 2      aphiaID  gbif_final   object  ogsl_final    int64  dtype mismatch
 3      aphiaID  gbif_final   object    cb_final  float64  dtype mismatch
 4      aphiaID  ogsl_final    int64    cb_final  float64  dtype mismatch
 ..         ...         ...      ...         ...      ...             ...
 183  waterBody  gbif_final      str  ogsl_final   object  dtype mismatch
 184  waterBody  gbif_final      str    cb_final   object  dtype mismatch
 185       year  obis_final  float64  ogsl_final   object  dtype mismatch
 186       year  gbif_final  float64  ogsl_f

In [13]:
# Analyze data type harmonization recommendations
type_mapping = {}

for col in sorted(all_columns):
    col_types = {}
    col_issues = []
    
    for dataset_name, df in parquets.items():
        if col in df.columns:
            col_types[dataset_name] = str(df[col].dtype)
    
    # Find discrepancies
    unique_types = set(col_types.values())
    
    if len(unique_types) > 1:
        col_issues.append(f"dtype mismatch: {', '.join(sorted(unique_types))}")
    
    # Check if column exists in all datasets
    missing_in = [d for d in parquets.keys() if col not in parquets[d].columns]
    if missing_in:
        col_issues.append(f"missing in: {', '.join(sorted(missing_in))}")
    
    # Recommend a unified type
    if col_types:
        # Get the most common type
        type_counts = {}
        for t in col_types.values():
            type_counts[t] = type_counts.get(t, 0) + 1
        most_common_dtype = max(type_counts, key=type_counts.get)
        
        # Map to standardized types
        dtype_str = most_common_dtype.lower()
        if 'int' in dtype_str:
            recommended = 'int64'
        elif 'float' in dtype_str or 'double' in dtype_str:
            recommended = 'float64'
        elif 'bool' in dtype_str:
            recommended = 'bool'
        elif any(x in dtype_str for x in ['datetime', 'timestamp']):
            recommended = 'datetime64[ns]'
        else:
            recommended = 'object (string)'
        
        type_mapping[col] = {
            'current_types': col_types,
            'issues': col_issues,
            'recommended_type': recommended
        }

# Create harmonization overview
harmonization_rows = []
for col, info in sorted(type_mapping.items()):
    current_types_str = "; ".join([f"{ds}: {dt}" for ds, dt in sorted(info['current_types'].items())])
    issues_str = "; ".join(info['issues']) if info['issues'] else "OK"
    
    harmonization_rows.append({
        'column': col,
        'current_types': current_types_str,
        'recommended_type': info['recommended_type'],
        'issues': issues_str
    })

harmonization_df = pd.DataFrame(harmonization_rows)

# Display summary by recommended type
print("\n" + "="*80)
print("DATA TYPE HARMONIZATION OVERVIEW")
print("="*80 + "\n")

for rec_type in sorted(set(harmonization_df['recommended_type'])):
    subset = harmonization_df[harmonization_df['recommended_type'] == rec_type]
    print(f"\n{rec_type.upper()}")
    print("-" * 80)
    for _, row in subset.iterrows():
        issues_display = "" if row['issues'] == "OK" else f" ⚠ {row['issues']}"
        print(f"  • {row['column']}{issues_display}")

# Save harmonization guide
harmonization_df.to_excel("data_type_harmonization_guide.xlsx", index=False, sheet_name="Harmonization Guide")
print(f"\n\n✓ Full harmonization guide saved to: data_type_harmonization_guide.xlsx")
print("\nFull table:")
harmonization_df


DATA TYPE HARMONIZATION OVERVIEW


FLOAT64
--------------------------------------------------------------------------------
  • coordinateUncertaintyInMeters ⚠ dtype mismatch: float64, object
  • decimalLatitudeStart
  • decimalLongitudeStart
  • depthA
  • depthB ⚠ dtype mismatch: float64, object
  • individualCount ⚠ dtype mismatch: float64, str
  • year ⚠ dtype mismatch: float64, object

INT64
--------------------------------------------------------------------------------
  • aphiaID ⚠ dtype mismatch: float64, int32, int64, object
  • sourceID ⚠ dtype mismatch: int64, object, str

OBJECT (STRING)
--------------------------------------------------------------------------------
  • ID
  • acceptedScientificName
  • areaSampled ⚠ dtype mismatch: object, str, string
  • areaSampledUnit ⚠ missing in: gbif_final, obis_final, ogsl_final
  • basisOfRecord ⚠ dtype mismatch: object, str
  • biomass ⚠ dtype mismatch: float64, object
  • biomassUnit ⚠ dtype mismatch: object, str
  • class
  •

,column,current_types,recommended_type,issues
0,ID,cb_final: object; gbif_final: object; obis_fin...,object (string),OK
1,acceptedScientificName,cb_final: str; gbif_final: str; obis_final: st...,object (string),OK
2,aphiaID,cb_final: float64; gbif_final: object; obis_fi...,int64,"dtype mismatch: float64, int32, int64, object"
3,areaSampled,cb_final: string; gbif_final: object; obis_fin...,object (string),"dtype mismatch: object, str, string"
4,areaSampledUnit,cb_final: str,object (string),"missing in: gbif_final, obis_final, ogsl_final"
...,...,...,...,...
67,validAuthority,cb_final: str; gbif_final: object; obis_final:...,object (string),"dtype mismatch: object, str"
68,verbatimID,cb_final: object; gbif_final: str; obis_final:...,object (string),"dtype mismatch: object, str"
69,verbatimScientificName,cb_final: str; gbif_final: str; obis_final: st...,object (string),"dtype mismatch: object, str"
70,waterBody,cb_final: object; gbif_final: str; obis_final:...,object (string),"dtype mismatch: object, str"
